# NSOS GPU Gold Validation — Tesla T4

This notebook is a fail-closed execution harness for the NSOS CUDA release gates. It requires a real Tesla T4, clones the exact validation commit from the private repository, builds the CUDA targets for `sm_75`, runs all GPU CTests, proves DP4A load/clone propagation, executes checkpoint v8 and FP16 training on GPU, runs Compute Sanitizer memcheck, and exports a SHA-256-addressed evidence bundle.

Before running, select **Runtime → Change runtime type → T4 GPU**. For the private clone, add a Colab secret named `GITHUB_TOKEN` with read access to the repository; if absent, the notebook asks for it without echoing.

In [ ]:
from __future__ import annotations

import base64
import getpass
import json
import os
import pathlib
import shutil
import subprocess
import sys

GPU_QUERY = [
    "nvidia-smi",
    "--query-gpu=name,compute_cap,driver_version",
    "--format=csv,noheader,nounits",
]
gpu_csv = subprocess.check_output(GPU_QUERY, text=True).strip()
rows = [[field.strip() for field in row.split(",")] for row in gpu_csv.splitlines() if row.strip()]
assert len(rows) == 1 and len(rows[0]) == 3, f"Expected exactly one CUDA GPU, got: {rows!r}"
gpu_name, compute_capability, driver_version = rows[0]
assert "Tesla T4" in gpu_name, f"Tesla T4 is required; active GPU is {gpu_name!r}"
assert tuple(map(int, compute_capability.split("."))) >= (7, 5), compute_capability
print(json.dumps({
    "gpu": gpu_name,
    "compute_capability": compute_capability,
    "driver": driver_version,
}, indent=2))


In [ ]:
REPOSITORY = "vitorGgC569/reimagined"
BRANCH = "codex-gpu-gold-validation"
CHECKOUT = pathlib.Path("/content/reimagined-gpu-gold")

if CHECKOUT.exists():
    raise RuntimeError(f"Refusing to reuse a possibly stale checkout: {CHECKOUT}")

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if not github_token:
    github_token = getpass.getpass("GitHub token (read-only, input hidden): ")
if not github_token:
    raise RuntimeError("A GitHub token is required for the private repository")

basic = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
clone = subprocess.run(
    [
        "git", "-c", f"http.extraHeader=AUTHORIZATION: basic {basic}",
        "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
        f"https://github.com/{REPOSITORY}.git", str(CHECKOUT),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
)
github_token = None
basic = None
if clone.returncode != 0:
    raise RuntimeError(f"Private clone failed with exit code {clone.returncode}")
subprocess.run(
    ["git", "remote", "set-url", "origin", f"https://github.com/{REPOSITORY}.git"],
    cwd=CHECKOUT,
    check=True,
)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, text=True).strip()
status = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"], cwd=CHECKOUT, text=True
).strip()
assert not status, f"Tracked checkout is dirty:\n{status}"
print(json.dumps({"branch": BRANCH, "commit": commit, "checkout": str(CHECKOUT)}, indent=2))


In [ ]:
NSOS = CHECKOUT / "OXN" / "nsos"
EVIDENCE = pathlib.Path("/content/nsos-gpu-gold-evidence")
BUILD = pathlib.Path("/content/nsos-build-gpu-gold-sm75")
command = [
    sys.executable,
    str(NSOS / "scripts" / "gpu_gold_validation.py"),
    "--source-dir", str(NSOS),
    "--build-dir", str(BUILD),
    "--output-dir", str(EVIDENCE),
    "--require-gpu-name", "Tesla T4",
    "--minimum-compute-capability", "7.5",
    "--cuda-architecture", "75",
    "--jobs", str(max(1, min(4, os.cpu_count() or 1))),
]
process = subprocess.Popen(
    command,
    cwd=CHECKOUT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
assert return_code == 0, f"GPU gold validation returned {return_code}; inspect {EVIDENCE}"


In [ ]:
result = json.loads((EVIDENCE / "result.json").read_text())
assert result["status"] == "pass", result.get("failure")
assert result["preflight"]["gpu"]["name"] == "Tesla T4"
assert result["gpu_ctest"] == {"tests": 24, "failures": 0, "disabled": 0, "skipped": 0}
assert len(result["compute_sanitizer"]) == 5
assert all(item["return_code"] == 0 for item in result["compute_sanitizer"].values())
assert all(item["error_count"] == 0 for item in result["compute_sanitizer"].values())
assert all(item["leaked_bytes"] == 0 for item in result["compute_sanitizer"].values())
mixed = result["contract_outputs"]["test_gpu_mixed_precision_contract"]
assert "fp16=executed" in mixed and "fp16=rejected_as_unsupported" not in mixed
checkpoint = result["contract_outputs"]["test_gpu_checkpoint_v8_continuation"]
assert "version=8" in checkpoint and "PASS" in checkpoint
dp4a = result["contract_outputs"]["test_gpu_model_pack_dp4a_lifecycle"]
assert "load_dispatches=1 clone_dispatches=1" in dp4a
print((EVIDENCE / "report.md").read_text())
print((EVIDENCE / "SHA256SUMS").read_text())


In [ ]:
archive_base = pathlib.Path("/content/nsos-gpu-gold-evidence")
archive_path = pathlib.Path(shutil.make_archive(str(archive_base), "zip", root_dir=EVIDENCE))
print(f"Evidence archive: {archive_path} ({archive_path.stat().st_size} bytes)")
from google.colab import files
files.download(str(archive_path))
